In [1]:
import os
import glob
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

def mhd_live_view(J, K, P, dt_out=0.5, poll_interval=3, field='rho'):
    """
    Live viewer — updates as new frames are written by CentPack.
    
    Parameters
    ----------
    field : str
        One of 'rho', 'p', 'jz', 'bz', 'b_on_rho'
    poll_interval : int
        Seconds between checks for new frames
    """
    sim_dir = '/Users/evandaniels/Research/CentPackL/simulation_MHD2d_reconnection_SD2'
    os.chdir(sim_dir)

    # Load mesh once
    y = np.loadtxt('mesh_files/y')
    x = np.array([])
    for p in range(P):
        xp = np.loadtxt(f'mesh_files/x_p{p:03d}')
        x = np.concatenate([x, xp])

    sub_sample = 16
    xx = x[::sub_sample]
    yy = y[::sub_sample]

    last_frame = -1

    print("Watching for frames... Interrupt kernel to stop.")

    while True:
        # Find latest available complete frame (all P ranks written)
        available = []
        n = 0
        while True:
            files = glob.glob(f'rho_files/rho_p*_{n:03d}')
            if len(files) == P:
                available.append(n)
                n += 1
            else:
                break

        if not available or available[-1] == last_frame:
            time.sleep(poll_interval)
            continue

        n = available[-1]
        last_frame = n
        t = n * dt_out

        # Load fields
        rho, b1, b3, press = [], [], [], []
        for p in range(P):
            rho.append(np.loadtxt(f'rho_files/rho_p{p:03d}_{n:03d}'))
            b1.append(np.loadtxt(f'b1_files/b1_p{p:03d}_{n:03d}'))
            b3.append(np.loadtxt(f'b3_files/b3_p{p:03d}_{n:03d}'))
            press.append(np.loadtxt(f'p_files/p_p{p:03d}_{n:03d}'))

        rho   = np.vstack(rho)
        b1    = np.vstack(b1)
        b3    = np.vstack(b3)
        press = np.vstack(press)

        bx = b1[::sub_sample, ::sub_sample]
        by = b3[::sub_sample, ::sub_sample]

        dx = x[1] - x[0]
        dy = y[1] - y[0]
        jz = np.gradient(b1, axis=1)/dy - np.gradient(b3, axis=0)/dx

        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Density + field lines
        cf0 = axes[0].contourf(x, y, rho.T, 20, cmap='viridis')
        axes[0].streamplot(xx, yy, bx.T, by.T, density=1,
                           color='white', linewidth=0.6)
        fig.colorbar(cf0, ax=axes[0], label='Density')
        axes[0].set_title(f'Density  t = {t:.2f}')
        axes[0].set_aspect('equal')

        # Jz
        lim = max(abs(jz.min()), abs(jz.max()))
        cf1 = axes[1].contourf(x, y, jz.T, 20, cmap='RdBu_r',
                                vmin=-lim, vmax=lim)
        axes[1].streamplot(xx, yy, bx.T, by.T, density=1,
                           color='black', linewidth=0.6)
        fig.colorbar(cf1, ax=axes[1], label='Jz')
        axes[1].set_title(f'Current Jz  t = {t:.2f}')
        axes[1].set_aspect('equal')

        # Bz out of plane
        lim = max(abs(b3.min()), abs(b3.max())) + 1e-10
        cf2 = axes[2].contourf(x, y, b3.T, 20, cmap='RdBu_r',
                                vmin=-lim, vmax=lim)
        fig.colorbar(cf2, ax=axes[2], label='Bz')
        axes[2].set_title(f'Bz (out-of-plane)  t = {t:.2f}')
        axes[2].set_aspect('equal')

        plt.tight_layout()
        display(fig)
        plt.close(fig)

        print(f'Frame {n} | t = {t:.2f} | press min = {press.min():.3e}')
        time.sleep(poll_interval)